# Leçon 13 - Mémoire de l'agent


## Configuration

Ce carnet montre comment créer un agent de réservation de voyage avec une **mémoire persistante** en utilisant le **Microsoft Agent Framework** (MAF).

Vous apprendrez comment les différents types de mémoire d’agent — de travail, à court terme et à long terme — influencent la manière dont un agent conserve et utilise les informations au fil des conversations.

**Prérequis :**
- Un projet Azure AI Foundry avec un modèle de chat déployé (par exemple `gpt-4o-mini`).
- Connexion via Azure CLI — exécutez `az login` dans votre terminal.
- `AZURE_AI_PROJECT_ENDPOINT` — le point de terminaison de votre projet Azure AI Foundry.
- `AZURE_AI_MODEL_DEPLOYMENT_NAME` — le nom de votre modèle déployé.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import json
from typing import Annotated
from datetime import datetime

from dotenv import load_dotenv

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

load_dotenv()

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ AzureAIProjectAgentProvider created")

## Types de mémoire des agents

Les agents d’IA peuvent utiliser différents types de mémoire, chacun remplissant une fonction distincte :

### Mémoire de travail
Le fil de la conversation lui-même — les messages échangés au cours d’une seule session. L’agent peut se référer aux messages précédents dans le même fil pour maintenir la cohérence. Dans MAF, cela est créé via **`agent.create_session()`**, qui retourne une `AgentSession`.

### Mémoire à court terme
Informations qui persistent pendant la durée d’une tâche ou d’une session mais ne sont pas stockées de manière permanente. Par exemple, l’agent peut accumuler des faits au cours d’une conversation pluri-tour de planification et les utiliser pour produire un itinéraire final.

### Mémoire à long terme
Préférences et faits qui persistent **entre les sessions**. Un utilisateur revenant ne devrait pas avoir à répéter ses restrictions alimentaires ou son style de voyage. La mémoire à long terme est généralement soutenue par un stockage externe — une base de données, un fichier ou un index vectoriel — et mise à disposition de l’agent via des outils.


## Mémoire de travail avec les sessions

La forme la plus simple de mémoire est la session de conversation. Lorsque vous passez la même session (créée via `agent.create_session()`) à des appels successifs de `agent.run()`, l'agent voit l'historique complet de cette conversation et peut se rappeler des détails précédents.

Créons un agent de voyage et démontrons la mémoire de travail.


In [ ]:
agent = await provider.create_agent(
    tools=[save_preference, get_preferences],
    name="TravelMemoryAgent",
    instructions=(
        "You are a travel agent who remembers user preferences across conversations. "
        "Track destinations mentioned, budget constraints, and travel dates."
    ),
)

session = agent.create_session()

# First message — the user shares preferences
response = await agent.run(
    "I love beach destinations and my budget is $3000",
    session=session,
)
print("Agent:", response)

# Second message — the agent should recall the budget from the thread
response = await agent.run(
    "What did I say my budget was?",
    session=session,
)
print("Agent:", response)

L'agent se souvenait correctement du budget parce que les deux messages partagent la même session. C'est une **mémoire de travail** — elle n'existe que pendant la durée de la session.

### Que se passe-t-il avec un nouveau fil ?

Si nous créons une **nouvelle** session, l'agent n'a aucun souvenir de la conversation précédente :


In [ ]:
new_session = agent.create_session()

response = await agent.run(
    "What is my budget?",
    session=new_session,
)
print("Agent:", response)
print("\n💡 The agent has no memory of the previous conversation — it's a fresh session.")

## Modèle de mémoire à long terme

Pour se souvenir des préférences de l’utilisateur **entre les sessions**, nous avons besoin d’un stockage persistant qui existe en dehors du fil de la conversation. L’agent accède à ce stockage via des **outils** — des fonctions qu’il peut appeler pour enregistrer et récupérer des informations.

Ci-dessous, nous implémentons un simple magasin de préférences en mémoire (en production, vous le soutiendriez avec une base de données ou un index vectoriel) et l’exposons comme des outils que l’agent peut utiliser.

### Architecture
```
┌─────────────────┐     ┌──────────────────┐     ┌─────────────────┐
│  MAF Agent      │────▶│  @tool functions  │────▶│  Preference     │
│  (LLM)          │     │  save / retrieve  │     │  Store (dict)   │
└─────────────────┘     └──────────────────┘     └─────────────────┘
         │                                                 │
    AgentSession                                   Persists across
    (working memory)                               sessions
```


In [ ]:
# --- Persistent preference store (simulated) ---
preference_store: dict[str, list[str]] = {}


@tool(approval_mode="never_require")
def save_preference(
    user_id: Annotated[str, "User identifier"],
    preference: Annotated[str, "A travel preference to remember"],
) -> str:
    """Save a user travel preference to long-term memory."""
    preference_store.setdefault(user_id, []).append(preference)
    return f"✅ Stored: {preference}"


@tool(approval_mode="never_require")
def get_preferences(
    user_id: Annotated[str, "User identifier"],
) -> str:
    """Retrieve all saved travel preferences for a user."""
    prefs = preference_store.get(user_id, [])
    if not prefs:
        return f"No saved preferences for {user_id}."
    return "Saved preferences:\n- " + "\n- ".join(prefs)


@tool(approval_mode="never_require")
def search_hotels(
    query: Annotated[str, "Search query — location, amenities, or tags"],
) -> str:
    """Search the hotel database for matching properties."""
    hotels = [
        {"name": "Le Meurice Paris", "location": "Paris, France", "price": 850, "tags": ["luxury", "romantic", "spa"]},
        {"name": "Four Seasons Maui", "location": "Maui, Hawaii", "price": 695, "tags": ["beach", "family", "resort"]},
        {"name": "Aman Tokyo", "location": "Tokyo, Japan", "price": 780, "tags": ["luxury", "city", "spa"]},
        {"name": "Hotel Sacher Vienna", "location": "Vienna, Austria", "price": 420, "tags": ["historic", "accessible", "cultural"]},
        {"name": "Fairmont Whistler", "location": "Whistler, Canada", "price": 380, "tags": ["ski", "family", "mountain"]},
    ]
    q = query.lower()
    matches = [
        h for h in hotels
        if q in h["name"].lower()
        or q in h["location"].lower()
        or any(q in t for t in h["tags"])
    ]
    if not matches:
        matches = hotels[:3]
    return json.dumps(matches, indent=2)


print("✅ Tools defined: save_preference, get_preferences, search_hotels")

### Scénario 1 — Un utilisateur pour la première fois réserve un voyage pour un anniversaire

Sarah visite pour la première fois. L'agent doit enregistrer ses préférences via les outils et les utiliser pour recommander des hôtels.


In [ ]:
travel_agent = await provider.create_agent(
    tools=[save_preference, get_preferences],
    name="TravelBookingAssistant",
    instructions=(
        "You are a personalized travel booking assistant with long-term memory.\n"
        "WORKFLOW:\n"
        "1. When a user starts a conversation, call get_preferences() to check for saved information.\n"
        "2. Store any new preferences the user mentions using save_preference().\n"
        "3. Use search_hotels() to find suitable options that match their preferences and budget.\n"
        "4. Do NOT recommend hotels that exceed the user's budget.\n\n"
        "IMPORTANT: Always use user_id='sarah_johnson_123' for all memory operations."
    ),
)

session_1 = travel_agent.create_session()

response = await travel_agent.run(
    "Hi! I'm Sarah and I'm planning a trip for my 10th wedding anniversary. "
    "We love romantic destinations, fine dining, and spa experiences. "
    "My husband has mobility issues, so we need accessible accommodations. "
    "Our budget is around $700-800 per night.",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
response = await travel_agent.run(
    "The Hotel Sacher sounds perfect! We're both vegetarian and I have a "
    "severe nut allergy. Can you note that for future trips?",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
# Verify what was stored
print("📋 Preference store contents:")
for uid, prefs in preference_store.items():
    print(f"\n  User: {uid}")
    for p in prefs:
        print(f"    - {p}")

### Scénario 2 — Sarah revient plusieurs semaines plus tard

Sarah commence un **tout nouveau fil de discussion** (simulant une nouvelle session). La mémoire de travail est vide, mais le magasin de préférences à long terme contient toujours ses informations. L'agent doit les récupérer et les utiliser pour personnaliser les recommandations.


In [ ]:
session_2 = travel_agent.create_session()  # New session — no working memory

response = await travel_agent.run(
    "Hi, my husband and I are planning another trip. Can you recommend a good hotel?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 The agent retrieved Sarah's saved preferences from long-term memory "
      "even though this is a completely new conversation thread.")

In [ ]:
response = await travel_agent.run(
    "Great suggestions! For the Maui option, what activities would you recommend for the kids?",
    session=session_2,
)
print("🤖 Agent:", response)

## Résumé

Dans cette leçon, vous avez appris les trois types de mémoire d'agent et comment les implémenter avec le Microsoft Agent Framework :

| Type de mémoire | Mécanisme MAF | Durée de vie |
|---|---|---|
| **Travail** | `agent.create_session()` | Conversation unique |
| **Court terme** | Contexte accumulé dans un fil | Tâche / session unique |
| **Long terme** | Stockage externe accessible via les fonctions `@tool` | Plusieurs sessions |

### Points clés
1. **`agent.create_session()`** fournit la mémoire de travail — l’agent voit l’intégralité de l’historique de la conversation dans une session.
2. **Les nouvelles sessions perdent le contexte** — sans mémoire à long terme, l’agent ne peut pas se souvenir des conversations passées.
3. **Les fonctions `@tool` comblent le fossé** — elles permettent à l’agent de sauvegarder et récupérer des informations depuis un stockage persistant.
4. **La personnalisation s’améliore avec le temps** — plus les préférences sont stockées, meilleures sont les recommandations de l’agent.

### Applications dans le monde réel
- **Service client** : se souvenir de l’historique et des préférences du client
- **Assistants personnels** : maintenir le contexte sur plusieurs jours ou semaines
- **Santé** : suivre les informations et préférences des patients
- **Commerce électronique** : shopping personnalisé basé sur l’historique

### Prochaines étapes
- Remplacer le dictionnaire en mémoire par une base de données ou un magasin vectoriel (ex. Azure AI Search)
- Ajouter une expiration de mémoire pour les informations sensibles au temps
- Construire des systèmes multi-agents avec mémoire partagée
- Explorer le carnet Cognee pour une mémoire soutenue par un graphe de connaissances


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Avertissement** :  
Ce document a été traduit à l’aide du service de traduction automatique [Co-op Translator](https://github.com/Azure/co-op-translator). Bien que nous nous efforcions d’assurer l’exactitude, veuillez noter que les traductions automatisées peuvent contenir des erreurs ou des inexactitudes. Le document original dans sa langue native doit être considéré comme la source faisant autorité. Pour les informations critiques, il est recommandé de recourir à une traduction professionnelle réalisée par un humain. Nous ne saurions être tenus responsables de toute méprise ou mauvaise interprétation résultant de l’utilisation de cette traduction.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
